# cv-data-lesson ライブラリ受け入れテスト

授業用Notebookとは別の確認用Notebookです。GitHubの固定コミットからライブラリをインストールし、データ取得・ラベル・split・bbox・CPU/GPU境界をまとめて確認します。データ取得セルは固定Releaseから約650MBを取得します。

In [ ]:
LIBRARY_REF = "5b3a2425eaf36caac0a0fd87893bd78d75a250da"
%pip install -q "cv-data-lesson[dataframe] @ git+https://github.com/708san/CV_data.git@5b3a2425eaf36caac0a0fd87893bd78d75a250da"

In [ ]:
import importlib.metadata as metadata
from pathlib import Path
import cvlesson
from cvlesson import datasets

installed_version = metadata.version("cv-data-lesson")
print("version:", installed_version)
print("module:", cvlesson.__file__)
assert cvlesson.__file__ is not None
assert LIBRARY_REF == "5b3a2425eaf36caac0a0fd87893bd78d75a250da"

In [ ]:
catalog = datasets.list_available()
catalog_ids = {item["id"] for item in catalog}
for required_id in ("geoset_ennea", "rdd2020_japan"):
    assert required_id in catalog_ids, required_id
print("catalog entries:", len(catalog), "required datasets: OK")

In [ ]:
# データをダウンロードせず、登録済みの元ラベルと日本語ラベルを確認する。
label_expectations = {
    "orange_fruit": {
        "cankered_orange": "傷んだオレンジ",
        "fresh_orange": "新鮮なオレンジ",
        "rotten_orange": "腐ったオレンジ",
    },
    "fruitseg30": {
        "Apple_Gala": "りんご（ガラ種）",
        "Apple_Golden Delicious": "りんご（ゴールデンデリシャス種）",
        "Green Coconut": "緑色のココナッツ",
        "Malta": "マルタオレンジ",
    },
    "indoor_waste_image": {"shell": "ナッツの殻"},
    "fire_recognition_image": {
        "Artificial_Fire": "擬似的な炎",
        "Fire": "炎",
        "Safe_Fire": "日常で扱われる火",
    },
    "fresh_rotten_fruits": {
        "fresh_apple": "新鮮なリンゴ",
        "rotten_apple": "傷んだリンゴ",
    },
    "agrifreshnet": {
        "Fresh Orange(1-9)": "新鮮なオレンジ",
        "Rotten Orange(20-35)": "傷んだオレンジ",
        "Semi fresh Orange(9-20)": "少し傷んだオレンジ",
    },
}
for dataset_id, expected in label_expectations.items():
    source = datasets._source(dataset_id)
    labels_ja = dict(zip(source.classes, source.class_names_ja))
    labels_ja.update(source.label_names_ja or {})
    for raw_label, expected_ja in expected.items():
        assert raw_label in source.classes, (dataset_id, raw_label)
        assert labels_ja[raw_label] == expected_ja, (dataset_id, raw_label, labels_ja[raw_label])
print("selected raw/Japanese labels: OK")

In [ ]:
CACHE_DIR = Path("/content/cvlesson-acceptance-cache")
geo = datasets.get("geoset_ennea", cache_dir=CACHE_DIR)
rdd = datasets.get("rdd2020_japan", cache_dir=CACHE_DIR)
print("GeoSetEnnea:", geo.splits)
print("RDD2020 Japan:", rdd.splits)
assert geo.splits == {"train": 84, "valid": 22, "test": 30}
assert rdd.splits == {"train": 6299, "valid": 802, "test": 799}
assert geo.root.is_dir() and rdd.root.is_dir()

In [ ]:
geo_df = geo.to_df()
rdd_df = rdd.to_df()
geo_raw_splits = set(geo_df["split"].unique())
rdd_raw_splits = set(rdd_df["split"].unique())
print("GeoSetEnnea raw splits:", geo_raw_splits)
print("RDD2020 raw splits:", rdd_raw_splits)
assert geo_raw_splits == {"train", "valid", "test"}
assert rdd_raw_splits <= {"train", "val", "valid", "test"}
assert rdd_raw_splits & {"val", "valid"}
assert {"class_name", "class_name_ja", "split", "boxes"} <= set(rdd_df.columns)
assert rdd_df["class_name"].notna().all()
assert rdd_df["class_name_ja"].notna().all()
print("DataFrame schema and original split notation: OK")

In [ ]:
# RDD2020の全bboxが有限値で、画像境界内（極小の丸め誤差を除く）にあることを確認する。
bbox_count = 0
for boxes in rdd_df["boxes"]:
    for box in boxes:
        bbox_count += 1
        values = [float(box[key]) for key in ("x_center", "y_center", "width", "height")]
        assert all(value == value and abs(value) != float("inf") for value in values)
        x_center, y_center, width, height = values
        assert -1e-9 <= x_center - width / 2 <= 1 + 1e-9
        assert -1e-9 <= y_center - height / 2 <= 1 + 1e-9
        assert -1e-9 <= x_center + width / 2 <= 1 + 1e-9
        assert -1e-9 <= y_center + height / 2 <= 1 + 1e-9
assert bbox_count == 16469
print("RDD2020 bbox count/range: OK (", bbox_count, ")")

In [ ]:
from cvlesson import generation, models

model_devices = models.available_devices()
print("ResNet devices:", model_devices)
assert model_devices["cpu"] is True
assert models.ResNetClassifier(device="cuda", weights=None).device == "cuda:0"
try:
    generation.load(device="cpu")
except generation.GenerationError as error:
    print("SDXL CPU boundary: OK -", error)
else:
    raise AssertionError("SDXLはCPUで実行可能と誤って判定されました")

In [ ]:
# PaddleOCR本体をダウンロードせず、WebPをPNGへデコードしてから渡す経路を確認する。
try:
    from PIL import Image
    import tempfile
    from cvlesson.ocr import OCR
except ImportError as error:
    print("OCR WebP smoke skipped:", error)
else:
    class ProbePipeline:
        def __init__(self):
            self.paths = []
        def predict(self, *, input):
            self.paths.append(input)
            return [{"res": {"rec_texts": ["検証"], "rec_scores": [0.99], "rec_polys": []}}]
    with tempfile.TemporaryDirectory() as temporary_directory:
        webp_path = Path(temporary_directory) / "sample.webp"
        Image.new("RGB", (32, 32), "white").save(webp_path, format="WEBP")
        probe = ProbePipeline()
        result = OCR(probe, device="cpu", language="japan").predict(webp_path)
        assert result.text == "検証"
        assert probe.paths and probe.paths[0].endswith("input.png")
    print("OCR WebP decode path: OK")

## 判定

ここまで全セルが成功すれば、指定コミットのライブラリをColabへ導入し、固定ReleaseのGeoSetEnnea/RDD2020、ラベル表記、元split保持、bbox検証、CPU/GPU境界、WebP入力経路を確認できています。ResNetの実学習やDETR/SAM/SDXLの重みを使う検証は、GPU利用量とモデル取得量が大きいため、この軽量受け入れテストから分離しています。